# Condition Number of Sampling Matrices for Hermite Functions

This notebook computes the 2-norm condition number $\kappa_2(A^{(d)})$ of both:
1. **Pure time sampling matrix** $A_{\mathrm{time}}^{(d)}$ (all $d$ samples in time domain)
2. **Mixed time-frequency sampling matrix** $A_{\mathrm{mixed}}^{(d)}$ (configurable split between time and frequency)

for the finite Hermite-function space $V_d = \mathrm{span}\{\varphi_0, \dots, \varphi_{d-1}\}$.

In [ ]:
import numpy as np
from scipy.linalg import svdvals
from scipy.sparse.linalg import svds
import matplotlib.pyplot as plt
import json
from pathlib import Path
import warnings
import os
warnings.filterwarnings('ignore')

# Create output folder for all results
os.makedirs('results/hermite_sampling_condition', exist_ok=True)

## Configuration Parameters

**`TIME_RATIO`**: Controls the fraction of samples allocated to the time domain in mixed sampling.
- `TIME_RATIO = 0.5` means $k = \lceil d/2 \rceil$ time samples, $m = d - k$ frequency samples (default)
- `TIME_RATIO = 2/3` means $k = \lceil 2d/3 \rceil$ time samples, $m = d - k$ frequency samples
- `TIME_RATIO = 1.0` would give all samples to time (same as pure time)
- `TIME_RATIO = 0.0` would give all samples to frequency

**`TIME_INTERVAL`**: The interval $[a, b]$ for time-domain sampling.
- Used for pure time sampling (all $d$ samples)
- Used for the time portion of mixed sampling ($k$ samples)

**`FREQ_INTERVAL`**: The interval $[a, b]$ for frequency-domain sampling.
- Used for the frequency portion of mixed sampling ($m$ samples)

The number of time samples is: $k = \lceil \texttt{TIME\_RATIO} \cdot d \rceil$, and frequency samples: $m = d - k$.

In [ ]:
# =============================================================================
# CONFIGURATION: Sampling parameters
# =============================================================================

# Time/Frequency ratio for mixed sampling
TIME_RATIO = 0.5  # Fraction of samples in time domain (0 to 1)
                  # 0.5 = half time, half freq (ceil for time if odd)
                  # 2/3 = ~2/3 time, ~1/3 freq
                  # 1.0 = all time (same as pure time)
                  # 0.0 = all frequency

# Sampling intervals
TIME_INTERVAL = (1.0, 2.0)  # [a, b] for time-domain samples
FREQ_INTERVAL = (-1.0, 0)  # [a, b] for frequency-domain samples

# =============================================================================
# Validation and summary
# =============================================================================
assert 0.0 <= TIME_RATIO <= 1.0, "TIME_RATIO must be between 0 and 1"
assert TIME_INTERVAL[0] < TIME_INTERVAL[1], "TIME_INTERVAL must have a < b"
assert FREQ_INTERVAL[0] < FREQ_INTERVAL[1], "FREQ_INTERVAL must have a < b"

print("Configuration Summary")
print("="*60)
print(f"TIME_RATIO = {TIME_RATIO}")
print(f"TIME_INTERVAL = [{TIME_INTERVAL[0]}, {TIME_INTERVAL[1]}]")
print(f"FREQ_INTERVAL = [{FREQ_INTERVAL[0]}, {FREQ_INTERVAL[1]}]")
print()
print("Example splits:")
for d in [10, 100, 500]:
    k = int(np.ceil(TIME_RATIO * d))
    m = d - k
    print(f"  d={d:4d}: k={k:4d} time samples in {TIME_INTERVAL}, m={m:4d} freq samples in {FREQ_INTERVAL}")

## 1. Define the sampling grids

For pure time sampling, use $d$ equispaced samples on `TIME_INTERVAL`.

For mixed sampling with configurable ratio:
- $k = \lceil \texttt{TIME\_RATIO} \cdot d \rceil$ time samples on `TIME_INTERVAL`
- $m = d - k$ frequency samples on `FREQ_INTERVAL`

In [ ]:
def build_grid(num_points, interval):
    """Build an equispaced grid on [a, b] with num_points points.
    
    Parameters:
        num_points: number of grid points
        interval: tuple (a, b) specifying the interval
    
    Returns:
        array of equispaced points
    """
    a, b = interval
    if num_points == 0:
        return np.array([])
    if num_points == 1:
        return np.array([(a + b) / 2.0])  # midpoint for single sample
    else:
        # j = 1, ..., num_points => indices 0, ..., num_points-1 in Python
        j = np.arange(1, num_points + 1)
        return a + (b - a) * (j - 1) / (num_points - 1)


def compute_time_freq_split(d, time_ratio):
    """Compute the number of time and frequency samples for given d and ratio.
    
    Parameters:
        d: total dimension
        time_ratio: fraction of samples in time domain (0 to 1)
    
    Returns:
        k: number of time samples (ceil of ratio * d)
        m: number of frequency samples (d - k)
    """
    k = int(np.ceil(time_ratio * d))
    # Ensure k is at least 0 and at most d
    k = max(0, min(d, k))
    m = d - k
    return k, m

## 2. Build Hermite function evaluation matrix using stable three-term recurrence

The normalized Hermite functions satisfy:
$$\varphi_0(x) = \pi^{-1/4} e^{-x^2/2}, \quad \varphi_1(x) = \sqrt{2} x \varphi_0(x)$$

and for $n \geq 1$:
$$\varphi_{n+1}(x) = \sqrt{\frac{2}{n+1}} x \varphi_n(x) - \sqrt{\frac{n}{n+1}} \varphi_{n-1}(x)$$

In [ ]:
def build_hermite_matrix(grid, d):
    """Build r x d matrix B where B[:,n] = phi_n(grid) for n=0,...,d-1.
    
    Uses stable three-term recurrence, vectorized over the grid.
    
    Parameters:
        grid: array of length r (evaluation points)
        d: number of Hermite functions (phi_0, ..., phi_{d-1})
    
    Returns:
        B: r x d real matrix
    """
    r = len(grid)
    if r == 0:
        return np.zeros((0, d), dtype=np.float64)
    
    B = np.zeros((r, d), dtype=np.float64)
    
    # phi_0(x) = pi^(-1/4) * exp(-x^2/2)
    phi_prev = np.pi**(-0.25) * np.exp(-grid**2 / 2)
    B[:, 0] = phi_prev
    
    if d == 1:
        return B
    
    # phi_1(x) = sqrt(2) * x * phi_0(x)
    phi_curr = np.sqrt(2.0) * grid * phi_prev
    B[:, 1] = phi_curr
    
    if d == 2:
        return B
    
    # Three-term recurrence for n = 1, 2, ..., d-2
    # phi_{n+1}(x) = sqrt(2/(n+1)) * x * phi_n(x) - sqrt(n/(n+1)) * phi_{n-1}(x)
    for n in range(1, d - 1):
        coeff1 = np.sqrt(2.0 / (n + 1))
        coeff2 = np.sqrt(n / (n + 1))
        phi_next = coeff1 * grid * phi_curr - coeff2 * phi_prev
        B[:, n + 1] = phi_next
        phi_prev = phi_curr
        phi_curr = phi_next
    
    return B

## 3. Build the sampling matrices

### (A) Pure time sampling matrix
$$A_{\mathrm{time}}^{(d)} \in \mathbb{R}^{d \times d}, \quad (A_{\mathrm{time}}^{(d)})_{j,n} = \varphi_{n-1}(t_j^{(d)})$$
where $t_j^{(d)}$ are $d$ equispaced points on `TIME_INTERVAL`.

### (B) Mixed time-frequency sampling matrix
With $k = \lceil \texttt{TIME\_RATIO} \cdot d \rceil$ time samples and $m = d - k$ frequency samples:
$$A_{\mathrm{mixed}}^{(d)} = \begin{pmatrix} B_t \\ B_\omega D \end{pmatrix}$$
where:
- $B_t$ uses $k$ samples on `TIME_INTERVAL`
- $B_\omega$ uses $m$ samples on `FREQ_INTERVAL`
- $D = \mathrm{diag}((-i)^0, (-i)^1, \ldots, (-i)^{d-1})$ applies the Fourier phase factors.

In [ ]:
def build_pure_time_matrix(d, time_interval):
    """Build the d x d pure time sampling matrix A_time^(d).
    
    A^(d)_{j,n} = phi_{n-1}(t^(d)_j), so column n contains phi_{n-1} evaluated on the grid.
    
    Parameters:
        d: dimension
        time_interval: tuple (a, b) for the sampling interval
    """
    t = build_grid(d, time_interval)
    return build_hermite_matrix(t, d)


def build_mixed_matrix(d, time_ratio, time_interval, freq_interval):
    """Build the d x d mixed time-frequency sampling matrix A_mixed^(d).
    
    Uses k = ceil(time_ratio * d) time samples and m = d - k frequency samples.
    
    A_mixed = [B_t; B_omega * D]
    where D = diag((-i)^0, (-i)^1, ..., (-i)^{d-1})
    
    Parameters:
        d: dimension
        time_ratio: fraction of samples in time domain (0 to 1)
        time_interval: tuple (a, b) for time-domain sampling
        freq_interval: tuple (a, b) for frequency-domain sampling
    
    Returns:
        A_mixed: d x d complex matrix
        k: number of time samples
        m: number of frequency samples
    """
    # Determine number of time and frequency samples
    k, m = compute_time_freq_split(d, time_ratio)
    
    # Build time grid and frequency grid
    t_grid = build_grid(k, time_interval)
    omega_grid = build_grid(m, freq_interval)
    
    # Build Hermite evaluation matrices
    B_t = build_hermite_matrix(t_grid, d)      # k x d real matrix
    B_omega = build_hermite_matrix(omega_grid, d)  # m x d real matrix
    
    # Build diagonal phase matrix D = diag((-i)^0, (-i)^1, ..., (-i)^{d-1})
    # (-i)^n = exp(-i * pi/2 * n)
    n_indices = np.arange(d)
    phase_factors = (-1j) ** n_indices  # 1, -i, -1, i, 1, -i, ...
    
    # Apply phase factors to frequency rows: B_omega @ D (right-multiply by diagonal)
    # This multiplies column n by (-i)^n
    B_omega_phased = B_omega * phase_factors  # broadcasting: m x d * d -> m x d
    
    # Stack time rows and frequency rows
    A_mixed = np.vstack([B_t, B_omega_phased]).astype(np.complex128)
    
    return A_mixed, k, m

## 4. Compute condition number

Use full SVD for $d \leq 2000$, and iterative methods for larger $d$.

In [ ]:
def compute_condition_number(A, d, svd_cutoff=2000):
    """Compute the 2-norm condition number of matrix A.
    
    Uses full SVD for d <= svd_cutoff, iterative SVD for larger d.
    Returns inf if sigma_min underflows to zero.
    """
    if d <= svd_cutoff:
        # Full SVD - O(d^3) but accurate
        try:
            s = svdvals(A)
            sigma_max = s[0]
            sigma_min = s[-1]
            if sigma_min == 0 or not np.isfinite(sigma_min):
                return np.inf, sigma_max, 0.0
            return sigma_max / sigma_min, sigma_max, sigma_min
        except Exception:
            return np.inf, np.nan, np.nan
    else:
        # Iterative SVD for large matrices
        try:
            # Get largest singular value
            _, s_max, _ = svds(A, k=1, which='LM', return_singular_vectors=True)
            sigma_max = s_max[0]
            
            # Get smallest singular value
            _, s_min, _ = svds(A, k=1, which='SM', return_singular_vectors=True)
            sigma_min = s_min[0]
            
            if sigma_min == 0 or not np.isfinite(sigma_min):
                return np.inf, sigma_max, 0.0
            return sigma_max / sigma_min, sigma_max, sigma_min
        except Exception:
            return np.inf, np.nan, np.nan

## 5. Define the set of dimensions to test

Using piecewise steps: all $d = 1, \dots, 50$; step by 5 up to 500.

In [ ]:
def get_test_dimensions():
    """Generate the set of dimensions to test."""
    dims = set()
    
    # All d = 1, ..., 50
    dims.update(range(1, 17))
    # dims.update(range(1, 21))
    
    # Step by 5 up to 500
    # dims.update(range(50, 501, 5))
    
    # # Ensure key values are included
    # key_values = {1, 2, 3, 5, 10, 20, 50, 100, 200, 500}
    # dims.update(key_values)
    
    return sorted(dims)

dimensions = get_test_dimensions()
print(f"Testing {len(dimensions)} dimensions from {min(dimensions)} to {max(dimensions)}")
print(f"First 20: {dimensions[:20]}")
print(f"Last 10: {dimensions[-10:]}")

## 6. Sanity checks for small dimensions

In [ ]:
print(f"Sanity checks for small dimensions (d <= 10):")
print(f"  TIME_RATIO = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL}")
print("="*95)
print(f"{'d':>3} | {'k':>3} | {'m':>3} | {'k/d':>5} | k+m=d | Time finite | Mixed finite | Time kappa | Mixed kappa")
print("-"*95)

for d in range(1, 11):
    # Pure time matrix
    A_time = build_pure_time_matrix(d, TIME_INTERVAL)
    time_finite = np.all(np.isfinite(A_time))
    kappa_time, _, _ = compute_condition_number(A_time, d)
    
    # Mixed matrix
    A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL)
    mixed_finite = np.all(np.isfinite(A_mixed))
    kappa_mixed, _, _ = compute_condition_number(A_mixed, d)
    
    # Verify k + m = d
    km_check = "PASS" if k + m == d else "FAIL"
    k_ratio = k / d if d > 0 else 0
    
    print(f"{d:3d} | {k:3d} | {m:3d} | {k_ratio:5.2f} | {km_check:5s} | {str(time_finite):11s} | {str(mixed_finite):12s} | {kappa_time:10.2e} | {kappa_mixed:10.2e}")

print(f"\nNote: k/d should be approximately {TIME_RATIO} (with ceiling rounding).")

In [ ]:
# Verify grids and phase factors for d=4
print(f"Verification of grids and phase factors for d=4:")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print("="*70)
d = 4
A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL)
print(f"k={k} time rows, m={m} frequency rows")

# Show grids
t_grid = build_grid(k, TIME_INTERVAL)
omega_grid = build_grid(m, FREQ_INTERVAL)
print(f"\nTime grid ({k} points on {TIME_INTERVAL}): {t_grid}")
print(f"Freq grid ({m} points on {FREQ_INTERVAL}): {omega_grid}")

print(f"\nPhase factors (-i)^n for n=0,1,2,3:")
for n in range(d):
    phase = (-1j)**n
    print(f"  n={n}: (-i)^{n} = {phase}")

print(f"\nMixed matrix A_mixed (shape {A_mixed.shape}):")
print(f"Time rows (first {k} rows, real):")
print(A_mixed[:k, :].real)
if m > 0:
    print(f"\nFrequency rows (last {m} rows, with phase factors):")
    print(A_mixed[k:, :])
else:
    print("\nNo frequency rows (all samples in time domain).")

## 7. Compute condition numbers for all dimensions

In [ ]:
results = []
output_file = Path("results/hermite_sampling_condition/hermite_condition_numbers.json")

print(f"Computing condition numbers...")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print("="*85)

for i, d in enumerate(dimensions):
    # Build pure time sampling matrix
    A_time = build_pure_time_matrix(d, TIME_INTERVAL)
    kappa_time, sigma_max_time, sigma_min_time = compute_condition_number(A_time, d)
    
    # Build mixed sampling matrix
    A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL)
    kappa_mixed, sigma_max_mixed, sigma_min_mixed = compute_condition_number(A_mixed, d)
    
    # Store results
    result = {
        "d": d,
        "time_ratio": TIME_RATIO,
        "time_interval": list(TIME_INTERVAL),
        "freq_interval": list(FREQ_INTERVAL),
        "k": k,
        "m": m,
        "kappa_time": float(kappa_time) if np.isfinite(kappa_time) else "inf",
        "kappa_mixed": float(kappa_mixed) if np.isfinite(kappa_mixed) else "inf",
        "sigma_max_time": float(sigma_max_time) if np.isfinite(sigma_max_time) else "nan",
        "sigma_min_time": float(sigma_min_time) if np.isfinite(sigma_min_time) else "nan",
        "sigma_max_mixed": float(sigma_max_mixed) if np.isfinite(sigma_max_mixed) else "nan",
        "sigma_min_mixed": float(sigma_min_mixed) if np.isfinite(sigma_min_mixed) else "nan",
        "svd_method": "full" if d <= 2000 else "iterative"
    }
    results.append(result)
    
    # Progress update
    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{kappa_time:.4e}" if np.isfinite(kappa_time) else "inf"
        km_str = f"{kappa_mixed:.4e}" if np.isfinite(kappa_mixed) else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d} (k={k}, m={m}): kappa_time = {kt_str}, kappa_mixed = {km_str}")
    
    # Save incrementally every 50 iterations
    if (i + 1) % 50 == 0:
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)

# Final save
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print("\nComputation complete!")
print(f"Results saved to {output_file}")

## 8. Save as CSV

In [ ]:
csv_file = Path("results/hermite_sampling_condition/hermite_condition_numbers.csv")

with open(csv_file, 'w') as f:
    f.write("d,time_ratio,time_interval_a,time_interval_b,freq_interval_a,freq_interval_b,k,m,kappa_time,kappa_mixed,sigma_max_time,sigma_min_time,sigma_max_mixed,sigma_min_mixed,svd_method\n")
    for r in results:
        f.write(f"{r['d']},{r['time_ratio']},{r['time_interval'][0]},{r['time_interval'][1]},{r['freq_interval'][0]},{r['freq_interval'][1]},")
        f.write(f"{r['k']},{r['m']},{r['kappa_time']},{r['kappa_mixed']},")
        f.write(f"{r['sigma_max_time']},{r['sigma_min_time']},{r['sigma_max_mixed']},{r['sigma_min_mixed']},{r['svd_method']}\n")

print(f"Results saved to {csv_file}")

## 9. Plot results: Pure Time vs Mixed Sampling

In [ ]:
# Extract data for plotting
d_vals = np.array([r['d'] for r in results])
kappa_time_vals = np.array([r['kappa_time'] if r['kappa_time'] != 'inf' else np.inf for r in results])
kappa_mixed_vals = np.array([r['kappa_mixed'] if r['kappa_mixed'] != 'inf' else np.inf for r in results])

# Filter out infinite values for plotting
time_finite_mask = np.isfinite(kappa_time_vals)
mixed_finite_mask = np.isfinite(kappa_mixed_vals)

d_time_finite = d_vals[time_finite_mask]
kappa_time_finite = kappa_time_vals[time_finite_mask]

d_mixed_finite = d_vals[mixed_finite_mask]
kappa_mixed_finite = kappa_mixed_vals[mixed_finite_mask]

print(f"Pure time: {len(d_time_finite)} finite out of {len(d_vals)}")
print(f"Mixed (TIME_RATIO={TIME_RATIO}): {len(d_mixed_finite)} finite out of {len(d_vals)}")

# Find where condition numbers become infinite
if np.any(~time_finite_mask):
    first_inf_time = d_vals[~time_finite_mask][0]
    print(f"Pure time becomes numerically infinite at d = {first_inf_time}")
else:
    first_inf_time = None
    
if np.any(~mixed_finite_mask):
    first_inf_mixed = d_vals[~mixed_finite_mask][0]
    print(f"Mixed becomes numerically infinite at d = {first_inf_mixed}")
else:
    first_inf_mixed = None

In [ ]:
# Create labels for sampling methods
time_label = f'One-sided reconstruction'

if TIME_RATIO == 0.5:
    mixed_label = f'Two-sided reconstruction'
else:
    from fractions import Fraction
    frac = Fraction(TIME_RATIO).limit_denominator(100)
    if frac.denominator <= 10:
        mixed_label = f'Mixed ({frac.numerator}/{frac.denominator} time {TIME_INTERVAL} + {frac.denominator - frac.numerator}/{frac.denominator} freq {FREQ_INTERVAL})'
    else:
        mixed_label = f'Mixed ({TIME_RATIO:.2f} time {TIME_INTERVAL} + {1-TIME_RATIO:.2f} freq {FREQ_INTERVAL})'

fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

# Plot 1: Linear x-axis, log y-axis
ax1.semilogy(d_time_finite, kappa_time_finite, 'b.-', markersize=8, linewidth=1.2, 
             label=time_label)
ax1.semilogy(d_mixed_finite, kappa_mixed_finite, 'r.-', markersize=8, linewidth=1.2,
             label=mixed_label)
ax1.set_xlabel('Dimension $N$', fontsize=22)
ax1.set_ylabel(r'Condition number', fontsize=22)
# ax1.set_title(f'Time interval [1.0,2.0], Frequency interval [-1.0, 0]', fontsize=13)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=13)
ax1.set_xlim([0, max(d_vals) * 1.02])

ax1.tick_params(axis='x', labelsize=18)  # x-axis tick label size
ax1.tick_params(axis='y', labelsize=12)  # y-axis tick label size

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to results/hermite_sampling_condition/hermite_condition_number_comparison.png")

## 10. Summary statistics

In [ ]:
print("Summary Statistics")
print("="*75)
print(f"Configuration:")
print(f"  TIME_RATIO = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL}")
print(f"Total dimensions tested: {len(d_vals)}")

print(f"\nPure Time Sampling (on {TIME_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_time_finite)}")
if len(kappa_time_finite) > 0:
    print(f"  Min kappa: {np.min(kappa_time_finite):.4e} at d = {d_time_finite[np.argmin(kappa_time_finite)]}")
    print(f"  Max kappa: {np.max(kappa_time_finite):.4e} at d = {d_time_finite[np.argmax(kappa_time_finite)]}")

print(f"\nMixed Sampling (time on {TIME_INTERVAL}, freq on {FREQ_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_mixed_finite)}")
if len(kappa_mixed_finite) > 0:
    print(f"  Min kappa: {np.min(kappa_mixed_finite):.4e} at d = {d_mixed_finite[np.argmin(kappa_mixed_finite)]}")
    print(f"  Max kappa: {np.max(kappa_mixed_finite):.4e} at d = {d_mixed_finite[np.argmax(kappa_mixed_finite)]}")

print(f"\nCondition numbers at key dimensions:")
print(f"{'d':>6} | {'k':>5} | {'m':>5} | {'kappa_time':>14} | {'kappa_mixed':>14} | {'Ratio (time/mixed)':>18}")
print("-"*80)
key_dims = [1, 2, 5, 10, 20, 50, 100, 200, 500]
for kd in key_dims:
    idx = np.where(d_vals == kd)[0]
    if len(idx) > 0:
        r = results[idx[0]]
        kt = kappa_time_vals[idx[0]]
        km = kappa_mixed_vals[idx[0]]
        kt_str = f"{kt:.4e}" if np.isfinite(kt) else "inf"
        km_str = f"{km:.4e}" if np.isfinite(km) else "inf"
        if np.isfinite(kt) and np.isfinite(km) and km > 0:
            ratio = kt / km
            ratio_str = f"{ratio:.2e}"
        else:
            ratio_str = "N/A"
        print(f"{kd:6d} | {r['k']:5d} | {r['m']:5d} | {kt_str:>14} | {km_str:>14} | {ratio_str:>18}")

## 11. Ratio analysis: How much better is mixed sampling?

In [ ]:
# Compute ratio where both are finite
both_finite = time_finite_mask & mixed_finite_mask
d_both = d_vals[both_finite]
ratio = kappa_time_vals[both_finite] / kappa_mixed_vals[both_finite]

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(d_both, ratio, 'g.-', markersize=3, linewidth=0.8)
ax.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ratio = 1 (equal)')
ax.set_xlabel('Dimension d', fontsize=12)
ax.set_ylabel(r'$\kappa_2(A_{\mathrm{time}}) / \kappa_2(A_{\mathrm{mixed}})$', fontsize=12)
ax.set_title(f'Condition Number Ratio: Pure Time / Mixed\n(TIME_RATIO={TIME_RATIO}, time={TIME_INTERVAL}, freq={FREQ_INTERVAL})', fontsize=12)
ax.grid(True, which='both', alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_ratio.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRatio statistics (pure time / mixed):")
print(f"  Configuration: TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print(f"  Min ratio: {np.min(ratio):.4e} at d = {d_both[np.argmin(ratio)]}")
print(f"  Max ratio: {np.max(ratio):.4e} at d = {d_both[np.argmax(ratio)]}")
print(f"  Mean ratio: {np.mean(ratio):.4e}")
print(f"  Median ratio: {np.median(ratio):.4e}")

# =============================================================================
# EXPERIMENT 1b: Regular Sampling with d-Dependent Intervals
# =============================================================================

The intervals now scale with dimension d:
- TIME_INTERVAL(d) = $[-\sqrt{2\pi d}, \sqrt{2\pi d}]$
- FREQ_INTERVAL(d) = $[-\sqrt{2\pi d}, \sqrt{2\pi d}]$

This captures the effective support of Hermite functions, which grows as $\sqrt{2n}$

In [ ]:
# =============================================================================
# EXPERIMENT 1b: Regular Sampling with d-Dependent Intervals
# =============================================================================

results_ddep = []
output_file_ddep = Path("results/hermite_sampling_condition/hermite_condition_numbers_ddep.json")

print("Computing condition numbers with d-DEPENDENT intervals...")
print(f"  Interval: [-sqrt(2*pi*d), sqrt(2*pi*d)]")
print(f"  TIME_RATIO={TIME_RATIO}")
print("="*75)

for i, d in enumerate(dimensions):
    # d-dependent interval: [-sqrt(2*pi*d), sqrt(2*pi*d)]
    interval_bound = np.sqrt(2 * np.pi * d)
    time_interval_d = (-interval_bound, interval_bound)
    freq_interval_d = (-interval_bound, interval_bound)

    # Build matrices with d-dependent intervals
    A_time = build_pure_time_matrix(d, time_interval_d)
    kappa_time, sigma_max_time, sigma_min_time = compute_condition_number(A_time, d)

    A_mixed, k, m = build_mixed_matrix(d, TIME_RATIO, time_interval_d, freq_interval_d)
    kappa_mixed, sigma_max_mixed, sigma_min_mixed = compute_condition_number(A_mixed, d)

    result = {
        "d": d,
        "interval_bound": float(interval_bound),
        "time_interval": list(time_interval_d),
        "freq_interval": list(freq_interval_d),
        "time_ratio": TIME_RATIO,
        "k": k,
        "m": m,
        "kappa_time": float(kappa_time) if np.isfinite(kappa_time) else "inf",
        "kappa_mixed": float(kappa_mixed) if np.isfinite(kappa_mixed) else "inf",
        "sigma_max_time": float(sigma_max_time) if np.isfinite(sigma_max_time) else "nan",
        "sigma_min_time": float(sigma_min_time) if np.isfinite(sigma_min_time) else "nan",
        "sigma_max_mixed": float(sigma_max_mixed) if np.isfinite(sigma_max_mixed) else "nan",
        "sigma_min_mixed": float(sigma_min_mixed) if np.isfinite(sigma_min_mixed) else "nan",
        "svd_method": "full" if d <= 2000 else "iterative"
    }
    results_ddep.append(result)

    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d}, interval=[+/-{interval_bound:.2f}]: "
              f"kappa_time={kappa_time:.4e}, kappa_mixed={kappa_mixed:.4e}")

    # Incremental saving
    if (i + 1) % 50 == 0:
        with open(output_file_ddep, 'w') as f:
            json.dump(results_ddep, f, indent=2)

# Final save
with open(output_file_ddep, 'w') as f:
    json.dump(results_ddep, f, indent=2)

print("="*75)
print(f"Saved results to {output_file_ddep}")

In [ ]:
# Extract d-dependent interval data for plotting
d_vals_ddep = np.array([r['d'] for r in results_ddep])
kappa_time_ddep = np.array([r['kappa_time'] if r['kappa_time'] != 'inf' else np.inf for r in results_ddep])
kappa_mixed_ddep = np.array([r['kappa_mixed'] if r['kappa_mixed'] != 'inf' else np.inf for r in results_ddep])

# Filter finite values
time_finite_mask_ddep = np.isfinite(kappa_time_ddep)
mixed_finite_mask_ddep = np.isfinite(kappa_mixed_ddep)

d_time_finite_ddep = d_vals_ddep[time_finite_mask_ddep]
kappa_time_finite_ddep = kappa_time_ddep[time_finite_mask_ddep]
d_mixed_finite_ddep = d_vals_ddep[mixed_finite_mask_ddep]
kappa_mixed_finite_ddep = kappa_mixed_ddep[mixed_finite_mask_ddep]

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.semilogy(d_time_finite_ddep, kappa_time_finite_ddep, 'b-o', linewidth=1.5, markersize=6,
            alpha=0.8, markeredgecolor='white', markeredgewidth=0.5, zorder=2,
            label='Pure Time')
ax.semilogy(d_mixed_finite_ddep, kappa_mixed_finite_ddep, 'r-s', linewidth=1.5, markersize=5,
            alpha=0.9, markeredgecolor='darkred', markeredgewidth=0.5, zorder=3,
            label='Mixed (time + freq)')

ax.set_xlabel('Dimension d', fontsize=12)
ax.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax.set_title(r'EXPERIMENT 1b: Regular Sampling with d-Dependent Intervals $[\pm\sqrt{2\pi d}]$', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim([0, max(dimensions) * 1.02])

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_ddep_regular.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved plot to results/hermite_sampling_condition/hermite_condition_number_ddep_regular.png")

# =============================================================================
# EXPERIMENT 2: Random Sampling (Uniform in Intervals)
# =============================================================================
# This section compares pure time vs mixed sampling with RANDOM sample locations
# (uniformly distributed in the intervals) instead of regular grids.
# The randomness between pure and mixed sampling cases is INDEPENDENT.
# Results are AVERAGED over NUM_RANDOM_TRIALS trials for robustness.

In [ ]:
# =============================================================================
# CONFIGURATION: Random Sampling Parameters
# =============================================================================

NUM_RANDOM_TRIALS = 10  # Number of random trials to average over
RANDOM_SEED = 42        # Base random seed for reproducibility

print(f"Random Sampling Configuration:")
print(f"  NUM_RANDOM_TRIALS = {NUM_RANDOM_TRIALS}")
print(f"  RANDOM_SEED = {RANDOM_SEED}")
print(f"  (Using same TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL})")

In [ ]:
def build_random_grid(num_points, interval, rng=None):
    """Build a random (uniform) grid on [a, b] with num_points points.
    
    Parameters:
        num_points: number of grid points
        interval: tuple (a, b) specifying the interval
        rng: numpy random generator (for reproducibility)
    
    Returns:
        array of uniformly distributed random points in [a, b]
    """
    a, b = interval
    if num_points == 0:
        return np.array([])
    if rng is None:
        rng = np.random.default_rng()
    return rng.uniform(a, b, size=num_points)


def build_pure_time_matrix_random(d, time_interval, rng=None):
    """Build the d x d pure time sampling matrix with RANDOM sample locations.
    
    Parameters:
        d: dimension
        time_interval: tuple (a, b) for the sampling interval
        rng: numpy random generator
    """
    t = build_random_grid(d, time_interval, rng)
    return build_hermite_matrix(t, d)


def build_mixed_matrix_random(d, time_ratio, time_interval, freq_interval, rng_time=None, rng_freq=None):
    """Build the d x d mixed sampling matrix with RANDOM sample locations.
    
    Uses k = ceil(time_ratio * d) time samples and m = d - k frequency samples.
    Time and frequency samples are drawn independently from uniform distributions.
    
    Parameters:
        d: dimension
        time_ratio: fraction of samples in time domain (0 to 1)
        time_interval: tuple (a, b) for time-domain sampling
        freq_interval: tuple (a, b) for frequency-domain sampling
        rng_time: random generator for time samples
        rng_freq: random generator for frequency samples
    
    Returns:
        A_mixed: d x d complex matrix
        k: number of time samples
        m: number of frequency samples
    """
    # Determine number of time and frequency samples
    k, m = compute_time_freq_split(d, time_ratio)
    
    # Build random time grid and frequency grid
    t_grid = build_random_grid(k, time_interval, rng_time)
    omega_grid = build_random_grid(m, freq_interval, rng_freq)
    
    # Build Hermite evaluation matrices
    B_t = build_hermite_matrix(t_grid, d)      # k x d real matrix
    B_omega = build_hermite_matrix(omega_grid, d)  # m x d real matrix
    
    # Build diagonal phase matrix D = diag((-i)^0, (-i)^1, ..., (-i)^{d-1})
    n_indices = np.arange(d)
    phase_factors = (-1j) ** n_indices
    
    # Apply phase factors to frequency rows
    B_omega_phased = B_omega * phase_factors
    
    # Stack time rows and frequency rows
    A_mixed = np.vstack([B_t, B_omega_phased]).astype(np.complex128)
    
    return A_mixed, k, m


print("Random sampling functions defined.")

In [ ]:
results_random = []
output_file_random = Path("results/hermite_sampling_condition/hermite_condition_numbers_random.json")

print(f"Computing condition numbers with RANDOM sampling...")
print(f"  Averaging over {NUM_RANDOM_TRIALS} trials (seed={RANDOM_SEED})")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}, FREQ_INTERVAL={FREQ_INTERVAL}")
print("="*90)

for i, d in enumerate(dimensions):
    k, m = compute_time_freq_split(d, TIME_RATIO)
    
    # Collect condition numbers across all trials
    kappa_time_trials = []
    kappa_mixed_trials = []
    
    for trial in range(NUM_RANDOM_TRIALS):
        # Create independent random generators for pure time and mixed sampling
        # Each trial gets different seeds, but pure and mixed are always independent
        trial_offset = trial * 100000
        
        # Pure time gets its own generator
        rng_pure = np.random.default_rng(RANDOM_SEED + d + trial_offset)
        
        # Mixed sampling gets separate generators for time and freq (independent from pure)
        rng_mixed_time = np.random.default_rng(RANDOM_SEED + d + trial_offset + 10000)
        rng_mixed_freq = np.random.default_rng(RANDOM_SEED + d + trial_offset + 20000)
        
        # Build pure time sampling matrix with random samples
        A_time_rand = build_pure_time_matrix_random(d, TIME_INTERVAL, rng_pure)
        kappa_time_rand, _, _ = compute_condition_number(A_time_rand, d)
        
        # Build mixed sampling matrix with random samples (independent randomness)
        A_mixed_rand, _, _ = build_mixed_matrix_random(d, TIME_RATIO, TIME_INTERVAL, FREQ_INTERVAL, 
                                                        rng_mixed_time, rng_mixed_freq)
        kappa_mixed_rand, _, _ = compute_condition_number(A_mixed_rand, d)
        
        # Store trial results (use inf for infinite values)
        kappa_time_trials.append(kappa_time_rand if np.isfinite(kappa_time_rand) else np.inf)
        kappa_mixed_trials.append(kappa_mixed_rand if np.isfinite(kappa_mixed_rand) else np.inf)
    
    # Convert to arrays
    kappa_time_trials = np.array(kappa_time_trials)
    kappa_mixed_trials = np.array(kappa_mixed_trials)
    
    # Compute statistics (mean, std, min, max) - only over finite values
    time_finite_trials = kappa_time_trials[np.isfinite(kappa_time_trials)]
    mixed_finite_trials = kappa_mixed_trials[np.isfinite(kappa_mixed_trials)]
    
    # Mean (use inf if all trials are inf)
    kappa_time_mean = np.mean(time_finite_trials) if len(time_finite_trials) > 0 else np.inf
    kappa_mixed_mean = np.mean(mixed_finite_trials) if len(mixed_finite_trials) > 0 else np.inf
    
    # Std
    kappa_time_std = np.std(time_finite_trials) if len(time_finite_trials) > 1 else 0.0
    kappa_mixed_std = np.std(mixed_finite_trials) if len(mixed_finite_trials) > 1 else 0.0
    
    # Min/Max
    kappa_time_min = np.min(time_finite_trials) if len(time_finite_trials) > 0 else np.inf
    kappa_time_max = np.max(time_finite_trials) if len(time_finite_trials) > 0 else np.inf
    kappa_mixed_min = np.min(mixed_finite_trials) if len(mixed_finite_trials) > 0 else np.inf
    kappa_mixed_max = np.max(mixed_finite_trials) if len(mixed_finite_trials) > 0 else np.inf
    
    # Store results
    result = {
        "d": d,
        "time_ratio": TIME_RATIO,
        "time_interval": list(TIME_INTERVAL),
        "freq_interval": list(FREQ_INTERVAL),
        "k": k,
        "m": m,
        "num_trials": NUM_RANDOM_TRIALS,
        "kappa_time_mean": float(kappa_time_mean) if np.isfinite(kappa_time_mean) else "inf",
        "kappa_time_std": float(kappa_time_std) if np.isfinite(kappa_time_std) else "nan",
        "kappa_time_min": float(kappa_time_min) if np.isfinite(kappa_time_min) else "inf",
        "kappa_time_max": float(kappa_time_max) if np.isfinite(kappa_time_max) else "inf",
        "kappa_mixed_mean": float(kappa_mixed_mean) if np.isfinite(kappa_mixed_mean) else "inf",
        "kappa_mixed_std": float(kappa_mixed_std) if np.isfinite(kappa_mixed_std) else "nan",
        "kappa_mixed_min": float(kappa_mixed_min) if np.isfinite(kappa_mixed_min) else "inf",
        "kappa_mixed_max": float(kappa_mixed_max) if np.isfinite(kappa_mixed_max) else "inf",
        "num_finite_time_trials": len(time_finite_trials),
        "num_finite_mixed_trials": len(mixed_finite_trials),
        "random_seed": RANDOM_SEED
    }
    results_random.append(result)
    
    # Progress update
    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{kappa_time_mean:.4e}" if np.isfinite(kappa_time_mean) else "inf"
        km_str = f"{kappa_mixed_mean:.4e}" if np.isfinite(kappa_mixed_mean) else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d} (k={k}, m={m}): mean kappa_time = {kt_str}, mean kappa_mixed = {km_str}")
    
    # Save incrementally every 50 iterations
    if (i + 1) % 50 == 0:
        with open(output_file_random, 'w') as f:
            json.dump(results_random, f, indent=2)

# Final save
with open(output_file_random, 'w') as f:
    json.dump(results_random, f, indent=2)

print("\nComputation complete!")
print(f"Results saved to {output_file_random}")

In [ ]:
# Extract random sampling data for plotting
# BACKWARDS COMPATIBLE: supports both old keys (kappa_time_random) and new keys (kappa_time_mean)

def get_random_value(r, new_key, old_key, default_val):
    """Get value from dict, trying new key first, then old key, then default."""
    if new_key in r:
        return r[new_key]
    elif old_key in r:
        return r[old_key]
    else:
        return default_val

# Use the same dimensions as defined in cell 13
d_vals_rand = np.array([r['d'] for r in results_random])

# Extract kappa values - try new keys first, fall back to old keys
kappa_time_rand_mean = np.array([
    get_random_value(r, 'kappa_time_mean', 'kappa_time_random', 'inf') 
    if get_random_value(r, 'kappa_time_mean', 'kappa_time_random', 'inf') != 'inf' 
    else np.inf 
    for r in results_random
])
kappa_mixed_rand_mean = np.array([
    get_random_value(r, 'kappa_mixed_mean', 'kappa_mixed_random', 'inf')
    if get_random_value(r, 'kappa_mixed_mean', 'kappa_mixed_random', 'inf') != 'inf'
    else np.inf
    for r in results_random
])

# Extract std values (only available in new format, default to 0 for old format)
kappa_time_rand_std = np.array([
    r.get('kappa_time_std', 0.0) if r.get('kappa_time_std', 'nan') != 'nan' else 0.0
    for r in results_random
])
kappa_mixed_rand_std = np.array([
    r.get('kappa_mixed_std', 0.0) if r.get('kappa_mixed_std', 'nan') != 'nan' else 0.0
    for r in results_random
])

# Filter out infinite values for plotting
time_rand_finite_mask = np.isfinite(kappa_time_rand_mean)
mixed_rand_finite_mask = np.isfinite(kappa_mixed_rand_mean)

d_time_rand_finite = d_vals_rand[time_rand_finite_mask]
kappa_time_rand_finite = kappa_time_rand_mean[time_rand_finite_mask]
kappa_time_rand_std_finite = kappa_time_rand_std[time_rand_finite_mask]

d_mixed_rand_finite = d_vals_rand[mixed_rand_finite_mask]
kappa_mixed_rand_finite = kappa_mixed_rand_mean[mixed_rand_finite_mask]
kappa_mixed_rand_std_finite = kappa_mixed_rand_std[mixed_rand_finite_mask]

# Determine if we have the new format with averaging
has_trials = 'num_trials' in results_random[0] if results_random else False
num_trials_used = results_random[0].get('num_trials', 1) if results_random else 1

print(f"Random Sampling Results{' (averaged over ' + str(num_trials_used) + ' trials)' if has_trials else ''}:")
print(f"  Pure time (random): {len(d_time_rand_finite)} finite out of {len(d_vals_rand)}")
print(f"  Mixed (random): {len(d_mixed_rand_finite)} finite out of {len(d_vals_rand)}")

In [ ]:
# Plot: Random Sampling - Pure Time vs Mixed (with error bands showing std if available)
fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

# Labels - adapt based on whether we have trial averaging
if has_trials and num_trials_used > 1:
    time_rand_label = f'Pure time RANDOM (mean ± std, n={num_trials_used})'
    mixed_rand_label = f'Mixed RANDOM (mean ± std, n={num_trials_used})'
else:
    time_rand_label = f'Pure time RANDOM'
    mixed_rand_label = f'Mixed RANDOM'

# Plot 1: Linear x-axis, log y-axis with error bands (if std available)
ax1.semilogy(d_time_rand_finite, kappa_time_rand_finite, 'b.-', markersize=3, linewidth=0.8, 
             label=time_rand_label)
if has_trials and np.any(kappa_time_rand_std_finite > 0):
    ax1.fill_between(d_time_rand_finite, 
                     np.maximum(kappa_time_rand_finite - kappa_time_rand_std_finite, 1e-10),
                     kappa_time_rand_finite + kappa_time_rand_std_finite,
                     alpha=0.2, color='blue')
ax1.semilogy(d_mixed_rand_finite, kappa_mixed_rand_finite, 'r.-', markersize=3, linewidth=0.8,
             label=mixed_rand_label)
if has_trials and np.any(kappa_mixed_rand_std_finite > 0):
    ax1.fill_between(d_mixed_rand_finite,
                     np.maximum(kappa_mixed_rand_finite - kappa_mixed_rand_std_finite, 1e-10),
                     kappa_mixed_rand_finite + kappa_mixed_rand_std_finite,
                     alpha=0.2, color='red')
ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
title_suffix = f' (avg over {num_trials_used} trials)' if has_trials and num_trials_used > 1 else ''
ax1.set_title(f'Random Sampling{title_suffix}\nPure Time vs Mixed', fontsize=13)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim([0, max(d_vals_rand) * 1.02])

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_random.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved to results/hermite_sampling_condition/hermite_condition_number_random.png")
if has_trials and num_trials_used > 1:
    print(f"(Shaded regions show ±1 standard deviation from {num_trials_used} trials)")

In [ ]:
# Plot all 4 curves: Regular and Random (mean) for both Pure Time and Mixed
# Uses dimensions from the global 'dimensions' variable defined in cell 13
fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

# Determine label suffix for random sampling
rand_label_suffix = f' (mean, n={num_trials_used})' if has_trials and num_trials_used > 1 else ''

# Plot 1: Linear x-axis, log y-axis
# Regular sampling
ax1.semilogy(d_time_finite, kappa_time_finite, 'b-', markersize=2, linewidth=1.2, 
             label=f'One-sided Regular')
ax1.semilogy(d_mixed_finite, kappa_mixed_finite, 'r-', markersize=2, linewidth=1.2,
             label=f'Two-sided Regular')
# Random sampling (mean)
ax1.semilogy(d_time_rand_finite, kappa_time_rand_finite, 'c--', markersize=2, linewidth=1.2, 
             label=f'One-sided Randomized')
ax1.semilogy(d_mixed_rand_finite, kappa_mixed_rand_finite, 'm--', markersize=2, linewidth=1.2,
             label=f'Two-sided Randomized')

ax1.set_xlabel('Dimension $N$', fontsize=21)
ax1.set_ylabel(r'Condition number', fontsize=21)
# ax1.set_title(f'Comparison on the interval $[-1,1]$ for both sides', fontsize=11)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=14)
ax1.set_xlim([0, max(dimensions) * 1.02])  # Use dimensions from cell 13

ax1.tick_params(axis='x', labelsize=18)  # x-axis tick label size
ax1.tick_params(axis='y', labelsize=12)  # y-axis tick label size

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_all_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to results/hermite_sampling_condition/hermite_condition_number_all_comparison.png")

In [ ]:
# Summary statistics for random sampling
# BACKWARDS COMPATIBLE with both old and new result formats
trial_info = f" (averaged over {num_trials_used} trials)" if has_trials and num_trials_used > 1 else ""
print(f"Summary Statistics: RANDOM Sampling{trial_info}")
print("="*85)
print(f"Configuration:")
print(f"  TIME_RATIO = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL}")
if has_trials:
    print(f"  NUM_RANDOM_TRIALS = {num_trials_used}")
    print(f"  RANDOM_SEED = {results_random[0].get('random_seed', 'N/A')}")
print(f"Total dimensions tested: {len(d_vals_rand)}")

print(f"\nPure Time RANDOM Sampling (on {TIME_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_time_rand_finite)}")
if len(kappa_time_rand_finite) > 0:
    label = "mean kappa" if has_trials else "kappa"
    print(f"  Min {label}: {np.min(kappa_time_rand_finite):.4e} at d = {d_time_rand_finite[np.argmin(kappa_time_rand_finite)]}")
    print(f"  Max {label}: {np.max(kappa_time_rand_finite):.4e} at d = {d_time_rand_finite[np.argmax(kappa_time_rand_finite)]}")

print(f"\nMixed RANDOM Sampling (time on {TIME_INTERVAL}, freq on {FREQ_INTERVAL}):")
print(f"  Finite condition numbers: {len(d_mixed_rand_finite)}")
if len(kappa_mixed_rand_finite) > 0:
    label = "mean kappa" if has_trials else "kappa"
    print(f"  Min {label}: {np.min(kappa_mixed_rand_finite):.4e} at d = {d_mixed_rand_finite[np.argmin(kappa_mixed_rand_finite)]}")
    print(f"  Max {label}: {np.max(kappa_mixed_rand_finite):.4e} at d = {d_mixed_rand_finite[np.argmax(kappa_mixed_rand_finite)]}")

# Comparison table - use key dimensions from dimensions list (cell 13) that actually exist
print(f"\nComparison at key dimensions (Random vs Regular):")
if has_trials:
    print(f"{'d':>6} | {'Pure REG':>12} | {'Pure RAND mean':>14} | {'Pure RAND std':>14} | {'Mixed REG':>12} | {'Mixed RAND mean':>15} | {'Mixed RAND std':>14}")
else:
    print(f"{'d':>6} | {'Pure REG':>12} | {'Pure RAND':>14} | {'Mixed REG':>12} | {'Mixed RAND':>14}")
print("-"*110)

# Use key dimensions that are in the global dimensions list
key_dims = [d for d in [1, 2, 5, 10, 20, 50, 100, 200, 500] if d in dimensions]
for kd in key_dims:
    idx = np.where(d_vals == kd)[0]
    idx_rand = np.where(d_vals_rand == kd)[0]
    if len(idx) > 0 and len(idx_rand) > 0:
        kt_reg = kappa_time_vals[idx[0]]
        km_reg = kappa_mixed_vals[idx[0]]
        kt_rand = kappa_time_rand_mean[idx_rand[0]]
        km_rand = kappa_mixed_rand_mean[idx_rand[0]]
        
        kt_reg_str = f"{kt_reg:.2e}" if np.isfinite(kt_reg) else "inf"
        km_reg_str = f"{km_reg:.2e}" if np.isfinite(km_reg) else "inf"
        kt_rand_str = f"{kt_rand:.2e}" if np.isfinite(kt_rand) else "inf"
        km_rand_str = f"{km_rand:.2e}" if np.isfinite(km_rand) else "inf"
        
        if has_trials:
            kt_rand_std_val = kappa_time_rand_std[idx_rand[0]]
            km_rand_std_val = kappa_mixed_rand_std[idx_rand[0]]
            kt_std_str = f"{kt_rand_std_val:.2e}" if np.isfinite(kt_rand_std_val) else "nan"
            km_std_str = f"{km_rand_std_val:.2e}" if np.isfinite(km_rand_std_val) else "nan"
            print(f"{kd:6d} | {kt_reg_str:>12} | {kt_rand_str:>14} | {kt_std_str:>14} | {km_reg_str:>12} | {km_rand_str:>15} | {km_std_str:>14}")
        else:
            print(f"{kd:6d} | {kt_reg_str:>12} | {kt_rand_str:>14} | {km_reg_str:>12} | {km_rand_str:>14}")

In [ ]:
# Compute ratio for random sampling where both are finite
both_rand_finite = time_rand_finite_mask & mixed_rand_finite_mask
d_both_rand = d_vals_rand[both_rand_finite]
ratio_rand = kappa_time_rand_mean[both_rand_finite] / kappa_mixed_rand_mean[both_rand_finite]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Determine label for random sampling ratio
rand_ratio_label = f'Random sampling (mean, n={num_trials_used})' if has_trials and num_trials_used > 1 else 'Random sampling'

# Plot 1: Random sampling ratio
ax1 = axes[0]
ax1.semilogy(d_both_rand, ratio_rand, 'm.-', markersize=3, linewidth=0.8, label=rand_ratio_label)
ax1.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ratio = 1 (equal)')
ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A_{\mathrm{time}}^{\mathrm{rand}}) / \kappa_2(A_{\mathrm{mixed}}^{\mathrm{rand}})$', fontsize=12)
title_suffix = ' (mean)' if has_trials else ''
ax1.set_title(f'Random Sampling: Pure Time / Mixed Ratio{title_suffix}', fontsize=12)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend()

# Plot 2: Compare regular vs random ratios
ax2 = axes[1]
ax2.semilogy(d_both, ratio, 'g-', markersize=2, linewidth=1.2, label='Regular sampling ratio')
ax2.semilogy(d_both_rand, ratio_rand, 'm--', markersize=2, linewidth=1.2, label=f'Random sampling ratio{" (mean)" if has_trials else ""}')
ax2.axhline(y=1, color='k', linestyle='--', alpha=0.5, label='Ratio = 1')
ax2.set_xlabel('Dimension d', fontsize=12)
ax2.set_ylabel(r'$\kappa_2(A_{\mathrm{time}}) / \kappa_2(A_{\mathrm{mixed}})$', fontsize=12)
ax2.set_title(f'Ratio Comparison: Regular vs Random', fontsize=12)
ax2.grid(True, which='both', alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_ratio_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

trial_info = f", averaged over {num_trials_used} trials" if has_trials and num_trials_used > 1 else ""
print(f"\nRatio statistics for RANDOM sampling (pure time / mixed){trial_info}:")
print(f"  Min ratio: {np.min(ratio_rand):.4e} at d = {d_both_rand[np.argmin(ratio_rand)]}")
print(f"  Max ratio: {np.max(ratio_rand):.4e} at d = {d_both_rand[np.argmax(ratio_rand)]}")
print(f"  Mean ratio: {np.mean(ratio_rand):.4e}")
print(f"  Median ratio: {np.median(ratio_rand):.4e}")

print(f"\nFor comparison, REGULAR sampling ratio statistics:")
print(f"  Min ratio: {np.min(ratio):.4e} at d = {d_both[np.argmin(ratio)]}")
print(f"  Max ratio: {np.max(ratio):.4e} at d = {d_both[np.argmax(ratio)]}")
print(f"  Mean ratio: {np.mean(ratio):.4e}")
print(f"  Median ratio: {np.median(ratio):.4e}")

## 12. Summary Plot: Condition Number vs Dimension d

A comprehensive visualization showing all condition number curves vs dimension d.

In [ ]:
# =============================================================================
# SUMMARY PLOT: Condition Number vs Dimension d
# =============================================================================
# This cell creates a comprehensive summary plot showing all condition number 
# curves as a function of dimension d.

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Determine label suffix for random sampling
rand_suffix = f' (mean, n={num_trials_used})' if has_trials and num_trials_used > 1 else ''

# -----------------------------------------------------------------------------
# Plot 1 (top-left): All 4 curves - Semi-log scale
# -----------------------------------------------------------------------------
ax1 = axes[0, 0]
ax1.semilogy(d_time_finite, kappa_time_finite, 'b-', linewidth=1.5, label='Pure time REGULAR')
ax1.semilogy(d_mixed_finite, kappa_mixed_finite, 'r-', linewidth=1.5, label='Mixed REGULAR')
ax1.semilogy(d_time_rand_finite, kappa_time_rand_finite, 'c--', linewidth=1.5, label=f'Pure time RANDOM{rand_suffix}')
ax1.semilogy(d_mixed_rand_finite, kappa_mixed_rand_finite, 'm--', linewidth=1.5, label=f'Mixed RANDOM{rand_suffix}')
ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax1.set_title('Condition Number vs d (Semi-log)', fontsize=13, fontweight='bold')
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=10)
ax1.set_xlim([0, max(dimensions) * 1.02])

# -----------------------------------------------------------------------------
# Plot 2 (top-right): All 4 curves - Log-log scale
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# Plot 3 (bottom-left): Regular sampling only with markers
# -----------------------------------------------------------------------------
ax3 = axes[1, 0]
ax3.semilogy(d_time_finite, kappa_time_finite, 'b.-', markersize=4, linewidth=1.0, label='Pure time REGULAR')
ax3.semilogy(d_mixed_finite, kappa_mixed_finite, 'r.-', markersize=4, linewidth=1.0, label='Mixed REGULAR')
ax3.set_xlabel('Dimension d', fontsize=12)
ax3.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax3.set_title('Regular Sampling: Condition Number vs d', fontsize=13, fontweight='bold')
ax3.grid(True, which='both', alpha=0.3)
ax3.legend(loc='upper left', fontsize=10)
ax3.set_xlim([0, max(dimensions) * 1.02])

# -----------------------------------------------------------------------------
# Plot 4 (bottom-right): Random sampling only with error bands
# -----------------------------------------------------------------------------
ax4 = axes[1, 1]
ax4.semilogy(d_time_rand_finite, kappa_time_rand_finite, 'c.-', markersize=4, linewidth=1.0, 
             label=f'Pure time RANDOM{rand_suffix}')
if has_trials and np.any(kappa_time_rand_std_finite > 0):
    ax4.fill_between(d_time_rand_finite, 
                     np.maximum(kappa_time_rand_finite - kappa_time_rand_std_finite, 1e-10),
                     kappa_time_rand_finite + kappa_time_rand_std_finite,
                     alpha=0.2, color='cyan')
ax4.semilogy(d_mixed_rand_finite, kappa_mixed_rand_finite, 'm.-', markersize=4, linewidth=1.0,
             label=f'Mixed RANDOM{rand_suffix}')
if has_trials and np.any(kappa_mixed_rand_std_finite > 0):
    ax4.fill_between(d_mixed_rand_finite,
                     np.maximum(kappa_mixed_rand_finite - kappa_mixed_rand_std_finite, 1e-10),
                     kappa_mixed_rand_finite + kappa_mixed_rand_std_finite,
                     alpha=0.2, color='magenta')
ax4.set_xlabel('Dimension d', fontsize=12)
ax4.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax4.set_title('Random Sampling: Condition Number vs d', fontsize=13, fontweight='bold')
ax4.grid(True, which='both', alpha=0.3)
ax4.legend(loc='upper left', fontsize=10)
ax4.set_xlim([0, max(dimensions) * 1.02])

# Add overall title with configuration info
fig.suptitle(f'Condition Number $\\kappa_2(A^{{(d)}})$ vs Dimension $d$\n'
             f'TIME_RATIO={TIME_RATIO}, time={TIME_INTERVAL}, freq={FREQ_INTERVAL}',
             fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSummary plot saved to results/hermite_sampling_condition/hermite_condition_number_summary.png")
print(f"\nConfiguration used:")
print(f"  TIME_RATIO = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  FREQ_INTERVAL = {FREQ_INTERVAL}")
print(f"  Dimensions tested: {len(dimensions)} (from {min(dimensions)} to {max(dimensions)})")
if has_trials:
    print(f"  Random trials: {num_trials_used}")

# =============================================================================
# EXPERIMENT 2b: Random Sampling with d-Dependent Intervals
# =============================================================================

Random sampling using d-dependent intervals:
- TIME_INTERVAL(d) = $[-\sqrt{2\pi d}, \sqrt{2\pi d}]$
- FREQ_INTERVAL(d) = $[-\sqrt{2\pi d}, \sqrt{2\pi d}]$

This captures the effective support of Hermite functions with random sample placement.

In [ ]:
# =============================================================================
# EXPERIMENT 2b: Random Sampling with d-Dependent Intervals
# =============================================================================

results_ddep_random = []
output_file_ddep_random = Path("results/hermite_sampling_condition/hermite_condition_numbers_ddep_random.json")

print("Computing condition numbers with RANDOM sampling and d-DEPENDENT intervals...")
print(f"  Interval: [-sqrt(2*pi*d), sqrt(2*pi*d)]")
print(f"  Averaging over {NUM_RANDOM_TRIALS} trials (seed={RANDOM_SEED})")
print(f"  TIME_RATIO={TIME_RATIO}")
print("="*75)

for i, d in enumerate(dimensions):
    # d-dependent interval: [-sqrt(2*pi*d), sqrt(2*pi*d)]
    interval_bound = np.sqrt(2 * np.pi * d)
    time_interval_d = (-interval_bound, interval_bound)
    freq_interval_d = (-interval_bound, interval_bound)

    k, m = compute_time_freq_split(d, TIME_RATIO)

    # Collect results across trials
    kappa_time_trials = []
    kappa_mixed_trials = []

    for trial in range(NUM_RANDOM_TRIALS):
        trial_offset = trial * 100000

        rng_pure = np.random.default_rng(RANDOM_SEED + d + trial_offset + 50000)
        rng_mixed_time = np.random.default_rng(RANDOM_SEED + d + trial_offset + 60000)
        rng_mixed_freq = np.random.default_rng(RANDOM_SEED + d + trial_offset + 70000)

        A_time_rand = build_pure_time_matrix_random(d, time_interval_d, rng_pure)
        kappa_time_rand, _, _ = compute_condition_number(A_time_rand, d)

        A_mixed_rand, _, _ = build_mixed_matrix_random(d, TIME_RATIO, time_interval_d, freq_interval_d,
                                                       rng_mixed_time, rng_mixed_freq)
        kappa_mixed_rand, _, _ = compute_condition_number(A_mixed_rand, d)

        kappa_time_trials.append(kappa_time_rand if np.isfinite(kappa_time_rand) else np.inf)
        kappa_mixed_trials.append(kappa_mixed_rand if np.isfinite(kappa_mixed_rand) else np.inf)

    kappa_time_trials = np.array(kappa_time_trials)
    kappa_mixed_trials = np.array(kappa_mixed_trials)

    time_finite = kappa_time_trials[np.isfinite(kappa_time_trials)]
    mixed_finite = kappa_mixed_trials[np.isfinite(kappa_mixed_trials)]

    kappa_time_mean = np.mean(time_finite) if len(time_finite) > 0 else np.inf
    kappa_time_std = np.std(time_finite) if len(time_finite) > 1 else 0.0
    kappa_mixed_mean = np.mean(mixed_finite) if len(mixed_finite) > 0 else np.inf
    kappa_mixed_std = np.std(mixed_finite) if len(mixed_finite) > 1 else 0.0

    result = {
        "d": d,
        "interval_bound": float(interval_bound),
        "k": k,
        "m": m,
        "num_trials": NUM_RANDOM_TRIALS,
        "kappa_time_mean": float(kappa_time_mean) if np.isfinite(kappa_time_mean) else "inf",
        "kappa_time_std": float(kappa_time_std),
        "kappa_mixed_mean": float(kappa_mixed_mean) if np.isfinite(kappa_mixed_mean) else "inf",
        "kappa_mixed_std": float(kappa_mixed_std),
    }
    results_ddep_random.append(result)

    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d}: time_mean={kappa_time_mean:.4e}, mixed_mean={kappa_mixed_mean:.4e}")

    if (i + 1) % 50 == 0:
        with open(output_file_ddep_random, 'w') as f:
            json.dump(results_ddep_random, f, indent=2)

with open(output_file_ddep_random, 'w') as f:
    json.dump(results_ddep_random, f, indent=2)

print("="*75)
print(f"Saved results to {output_file_ddep_random}")

In [ ]:
# Extract d-dependent random sampling data for plotting
d_vals_ddep_rand = np.array([r['d'] for r in results_ddep_random])
kappa_time_ddep_rand_mean = np.array([r['kappa_time_mean'] if r['kappa_time_mean'] != 'inf' else np.inf for r in results_ddep_random])
kappa_mixed_ddep_rand_mean = np.array([r['kappa_mixed_mean'] if r['kappa_mixed_mean'] != 'inf' else np.inf for r in results_ddep_random])
kappa_time_ddep_rand_std = np.array([r['kappa_time_std'] for r in results_ddep_random])
kappa_mixed_ddep_rand_std = np.array([r['kappa_mixed_std'] for r in results_ddep_random])

time_finite_mask_ddep_rand = np.isfinite(kappa_time_ddep_rand_mean)
mixed_finite_mask_ddep_rand = np.isfinite(kappa_mixed_ddep_rand_mean)

d_time_finite_ddep_rand = d_vals_ddep_rand[time_finite_mask_ddep_rand]
kappa_time_finite_ddep_rand = kappa_time_ddep_rand_mean[time_finite_mask_ddep_rand]
kappa_time_std_finite_ddep_rand = kappa_time_ddep_rand_std[time_finite_mask_ddep_rand]

d_mixed_finite_ddep_rand = d_vals_ddep_rand[mixed_finite_mask_ddep_rand]
kappa_mixed_finite_ddep_rand = kappa_mixed_ddep_rand_mean[mixed_finite_mask_ddep_rand]
kappa_mixed_std_finite_ddep_rand = kappa_mixed_ddep_rand_std[mixed_finite_mask_ddep_rand]

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.semilogy(d_time_finite_ddep_rand, kappa_time_finite_ddep_rand, 'c-o', linewidth=1.5, markersize=6,
            alpha=0.8, markeredgecolor='white', markeredgewidth=0.5, zorder=2,
            label=f'Pure Time (mean, n={NUM_RANDOM_TRIALS})')
if np.any(kappa_time_std_finite_ddep_rand > 0):
    ax.fill_between(d_time_finite_ddep_rand,
                    np.maximum(kappa_time_finite_ddep_rand - kappa_time_std_finite_ddep_rand, 1e-10),
                    kappa_time_finite_ddep_rand + kappa_time_std_finite_ddep_rand,
                    alpha=0.2, color='cyan')

ax.semilogy(d_mixed_finite_ddep_rand, kappa_mixed_finite_ddep_rand, 'm-s', linewidth=1.5, markersize=5,
            alpha=0.9, markeredgecolor='purple', markeredgewidth=0.5, zorder=3,
            label=f'Mixed (mean, n={NUM_RANDOM_TRIALS})')
if np.any(kappa_mixed_std_finite_ddep_rand > 0):
    ax.fill_between(d_mixed_finite_ddep_rand,
                    np.maximum(kappa_mixed_finite_ddep_rand - kappa_mixed_std_finite_ddep_rand, 1e-10),
                    kappa_mixed_finite_ddep_rand + kappa_mixed_std_finite_ddep_rand,
                    alpha=0.2, color='magenta')

ax.set_xlabel('Dimension d', fontsize=12)
ax.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax.set_title(r'EXPERIMENT 2b: Random Sampling with d-Dependent Intervals $[\pm\sqrt{2\pi d}]$', fontsize=13)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim([0, max(dimensions) * 1.02])

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_ddep_random.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved plot to results/hermite_sampling_condition/hermite_condition_number_ddep_random.png")

In [ ]:
# =============================================================================
# Comparison: Fixed Intervals vs d-Dependent Intervals
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
ax1.semilogy(d_time_finite, kappa_time_finite, 'b-', linewidth=1.5, alpha=0.7,
             label='Pure Time (fixed interval)')
ax1.semilogy(d_mixed_finite, kappa_mixed_finite, 'r-', linewidth=1.5, alpha=0.7,
             label='Mixed (fixed interval)')
ax1.semilogy(d_time_finite_ddep, kappa_time_finite_ddep, 'b--o', linewidth=1.5, markersize=4,
             alpha=0.9, markeredgecolor='white', markeredgewidth=0.5,
             label='Pure Time (d-dependent)')
ax1.semilogy(d_mixed_finite_ddep, kappa_mixed_finite_ddep, 'r--s', linewidth=1.5, markersize=4,
             alpha=0.9, markeredgecolor='darkred', markeredgewidth=0.5,
             label='Mixed (d-dependent)')

ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax1.set_title('Regular Sampling: Fixed vs d-Dependent Intervals', fontsize=12)
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, which='both', alpha=0.3)
ax1.set_xlim([0, max(dimensions) * 1.02])

ax2 = axes[1]
ax2.semilogy(d_time_rand_finite, kappa_time_rand_finite, 'c-', linewidth=1.5, alpha=0.7,
             label='Pure Time (fixed, mean)')
ax2.semilogy(d_mixed_rand_finite, kappa_mixed_rand_finite, 'm-', linewidth=1.5, alpha=0.7,
             label='Mixed (fixed, mean)')
ax2.semilogy(d_time_finite_ddep_rand, kappa_time_finite_ddep_rand, 'c--o', linewidth=1.5, markersize=4,
             alpha=0.9, markeredgecolor='white', markeredgewidth=0.5,
             label='Pure Time (d-dep, mean)')
ax2.semilogy(d_mixed_finite_ddep_rand, kappa_mixed_finite_ddep_rand, 'm--s', linewidth=1.5, markersize=4,
             alpha=0.9, markeredgecolor='purple', markeredgewidth=0.5,
             label='Mixed (d-dep, mean)')

ax2.set_xlabel('Dimension d', fontsize=12)
ax2.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax2.set_title('Random Sampling: Fixed vs d-Dependent Intervals', fontsize=12)
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, which='both', alpha=0.3)
ax2.set_xlim([0, max(dimensions) * 1.02])

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_ddep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved comparison plot to results/hermite_sampling_condition/hermite_condition_number_ddep_comparison.png")

# =============================================================================
# EXPERIMENT 3: DFT-Stabilizer
# =============================================================================
# This experiment uses a DFT-based frequency conversion instead of separate
# frequency-domain sampling points. All d samples are taken in TIME_INTERVAL,
# then a subset (m samples) are transformed to frequency domain via the DFT matrix.
#
# Key differences from previous experiments:
# - Only TIME_INTERVAL is used (no separate FREQ_INTERVAL)
# - All d sample points are in the time domain
# - The "frequency samples" are obtained by applying an m×m DFT matrix to m of the rows
# - Regular case: alternating selection (even indices → time, odd indices → freq)
# - Random case: random selection of which samples get DFT-transformed

In [ ]:
# =============================================================================
# DFT-Stabilizer Functions
# =============================================================================

def build_dft_matrix(m):
    """Build the m×m unitary DFT matrix.
    
    F[j,l] = (1/sqrt(m)) * exp(-2*pi*i*j*l/m)
    This is unitary: F @ F.conj().T = I
    
    Parameters:
        m: size of the DFT matrix
    
    Returns:
        F: m×m complex unitary DFT matrix
    """
    if m == 0:
        return np.zeros((0, 0), dtype=np.complex128)
    j = np.arange(m).reshape(-1, 1)
    l = np.arange(m).reshape(1, -1)
    F = np.exp(-2j * np.pi * j * l / m) / np.sqrt(m)
    return F


def build_dft_stabilized_matrix(d, time_ratio, time_interval):
    """Build DFT-stabilized mixed matrix (REGULAR sampling).
    
    Steps:
    1. Build d×d Hermite matrix on d equispaced points in time_interval
    2. Select k rows for time (even indices: 0, 2, 4, ...)
    3. Select m rows for DFT transform (odd indices: 1, 3, 5, ...)
    4. Apply m×m DFT matrix to the m frequency rows
    5. Stack: A = [B_time; F_m @ B_freq]
    
    Parameters:
        d: dimension
        time_ratio: fraction of samples in time domain
        time_interval: tuple (a, b) for sampling interval
    
    Returns:
        A_mixed: d×d complex matrix
        k: number of time samples
        m: number of frequency samples (DFT-transformed)
    """
    k, m = compute_time_freq_split(d, time_ratio)
    
    # All d samples in time interval (equispaced)
    t_grid = build_grid(d, time_interval)
    B_full = build_hermite_matrix(t_grid, d)  # d×d
    
    if m == 0:
        # All samples are time samples, no DFT needed
        return B_full.astype(np.complex128), k, m
    
    # Alternating selection: even indices for time, odd for freq
    all_indices = np.arange(d)
    even_indices = all_indices[0::2]  # 0, 2, 4, ...
    odd_indices = all_indices[1::2]   # 1, 3, 5, ...
    
    # Take first k from even (time) and first m from odd (freq)
    # If not enough even/odd, wrap around
    if len(even_indices) >= k:
        time_indices = even_indices[:k]
    else:
        time_indices = np.concatenate([even_indices, odd_indices[:k-len(even_indices)]])
    
    if len(odd_indices) >= m:
        freq_indices = odd_indices[:m]
    else:
        # Take remaining from even indices that weren't used for time
        remaining_even = even_indices[k:] if k < len(even_indices) else np.array([], dtype=int)
        freq_indices = np.concatenate([odd_indices, remaining_even[:m-len(odd_indices)]])
    
    # Ensure we have exactly k time and m freq indices
    time_indices = time_indices[:k]
    freq_indices = freq_indices[:m]
    
    B_time = B_full[time_indices, :]  # k×d
    B_freq_samples = B_full[freq_indices, :]  # m×d
    
    # Apply DFT to frequency rows
    F_m = build_dft_matrix(m)  # m×m unitary
    B_freq_dft = F_m @ B_freq_samples  # m×d (complex)
    
    # Stack time rows and DFT-transformed frequency rows
    A_mixed = np.vstack([B_time, B_freq_dft]).astype(np.complex128)
    
    return A_mixed, k, m


def build_dft_stabilized_matrix_random(d, time_ratio, time_interval, rng=None):
    """Build DFT-stabilized mixed matrix (RANDOM sampling).
    
    Steps:
    1. Build d×d Hermite matrix on d RANDOM points in time_interval
    2. Randomly select k indices for time samples
    3. Remaining m indices get DFT transformed
    4. Apply m×m DFT matrix to the m frequency rows
    5. Stack: A = [B_time; F_m @ B_freq]
    
    Parameters:
        d: dimension
        time_ratio: fraction of samples in time domain
        time_interval: tuple (a, b) for sampling interval
        rng: numpy random generator
    
    Returns:
        A_mixed: d×d complex matrix
        k: number of time samples
        m: number of frequency samples (DFT-transformed)
    """
    if rng is None:
        rng = np.random.default_rng()
    
    k, m = compute_time_freq_split(d, time_ratio)
    
    # All d samples in time interval (RANDOM positions)
    t_grid = build_random_grid(d, time_interval, rng)
    B_full = build_hermite_matrix(t_grid, d)  # d×d
    
    if m == 0:
        # All samples are time samples, no DFT needed
        return B_full.astype(np.complex128), k, m
    
    # Randomly select which samples become time vs frequency
    all_indices = np.arange(d)
    rng.shuffle(all_indices)
    time_indices = np.sort(all_indices[:k])
    freq_indices = np.sort(all_indices[k:])
    
    B_time = B_full[time_indices, :]  # k×d
    B_freq_samples = B_full[freq_indices, :]  # m×d
    
    # Apply DFT to frequency rows
    F_m = build_dft_matrix(m)  # m×m unitary
    B_freq_dft = F_m @ B_freq_samples  # m×d (complex)
    
    # Stack time rows and DFT-transformed frequency rows
    A_mixed = np.vstack([B_time, B_freq_dft]).astype(np.complex128)
    
    return A_mixed, k, m


# Verify DFT matrix is unitary
print("DFT-Stabilizer functions defined.")
print("\nVerifying DFT matrix is unitary:")
for m in [4, 5, 10]:
    F = build_dft_matrix(m)
    identity_check = F @ F.conj().T
    max_error = np.max(np.abs(identity_check - np.eye(m)))
    print(f"  m={m}: max|F @ F^H - I| = {max_error:.2e} {'✓' if max_error < 1e-10 else '✗'}")

In [ ]:
# =============================================================================
# EXPERIMENT 3a: DFT-Stabilizer with REGULAR Sampling
# =============================================================================

results_dft = []
output_file_dft = Path("results/hermite_sampling_condition/hermite_condition_numbers_dft.json")

print(f"EXPERIMENT 3a: DFT-Stabilizer (REGULAR sampling)")
print(f"Computing condition numbers...")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}")
print(f"  Using alternating selection: even indices → time, odd indices → DFT")
print("="*85)

for i, d in enumerate(dimensions):
    # Build pure time sampling matrix (reuse existing function)
    A_time = build_pure_time_matrix(d, TIME_INTERVAL)
    kappa_time, sigma_max_time, sigma_min_time = compute_condition_number(A_time, d)
    
    # Build DFT-stabilized mixed matrix
    A_dft, k, m = build_dft_stabilized_matrix(d, TIME_RATIO, TIME_INTERVAL)
    kappa_dft, sigma_max_dft, sigma_min_dft = compute_condition_number(A_dft, d)
    
    # Store results
    result = {
        "d": d,
        "time_ratio": TIME_RATIO,
        "time_interval": list(TIME_INTERVAL),
        "k": k,
        "m": m,
        "kappa_time": float(kappa_time) if np.isfinite(kappa_time) else "inf",
        "kappa_dft": float(kappa_dft) if np.isfinite(kappa_dft) else "inf",
        "sigma_max_time": float(sigma_max_time) if np.isfinite(sigma_max_time) else "nan",
        "sigma_min_time": float(sigma_min_time) if np.isfinite(sigma_min_time) else "nan",
        "sigma_max_dft": float(sigma_max_dft) if np.isfinite(sigma_max_dft) else "nan",
        "sigma_min_dft": float(sigma_min_dft) if np.isfinite(sigma_min_dft) else "nan",
    }
    results_dft.append(result)
    
    # Progress update
    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{kappa_time:.4e}" if np.isfinite(kappa_time) else "inf"
        kd_str = f"{kappa_dft:.4e}" if np.isfinite(kappa_dft) else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d} (k={k}, m={m}): kappa_time = {kt_str}, kappa_dft = {kd_str}")

# Save results
with open(output_file_dft, 'w') as f:
    json.dump(results_dft, f, indent=2)

print("\nComputation complete!")
print(f"Results saved to {output_file_dft}")

In [ ]:
# Extract DFT-stabilized data for plotting
d_vals_dft = np.array([r['d'] for r in results_dft])
kappa_time_dft_vals = np.array([r['kappa_time'] if r['kappa_time'] != 'inf' else np.inf for r in results_dft])
kappa_dft_vals = np.array([r['kappa_dft'] if r['kappa_dft'] != 'inf' else np.inf for r in results_dft])

# Filter out infinite values for plotting
time_dft_finite_mask = np.isfinite(kappa_time_dft_vals)
dft_finite_mask = np.isfinite(kappa_dft_vals)

d_time_dft_finite = d_vals_dft[time_dft_finite_mask]
kappa_time_dft_finite = kappa_time_dft_vals[time_dft_finite_mask]

d_dft_finite = d_vals_dft[dft_finite_mask]
kappa_dft_finite = kappa_dft_vals[dft_finite_mask]

print(f"DFT-Stabilizer (REGULAR) Results:")
print(f"  Pure time: {len(d_time_dft_finite)} finite out of {len(d_vals_dft)}")
print(f"  DFT-stabilized: {len(d_dft_finite)} finite out of {len(d_vals_dft)}")

# Plot comparison
fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

# Plot 1: Semi-log scale
ax1.semilogy(d_time_dft_finite, kappa_time_dft_finite, 'b-o', markersize=7, linewidth=1.5,
             alpha=0.8, markeredgecolor='white', markeredgewidth=0.5, zorder=2, 
             label=f'Pure time REGULAR')
ax1.semilogy(d_dft_finite, kappa_dft_finite, 'g-s', markersize=5, linewidth=1.5,
             alpha=0.9, markeredgecolor='darkgreen', markeredgewidth=0.5, zorder=3,
             label=f'DFT-stabilized REGULAR')
ax1.set_xlabel('Dimension d', fontsize=12)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=12)
ax1.set_title(f'EXPERIMENT 3a: DFT-Stabilizer (Regular)\nTIME_RATIO={TIME_RATIO}, interval={TIME_INTERVAL}', fontsize=12)
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=10)
ax1.set_xlim([0, max(dimensions) * 1.02])

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_dft_regular.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to results/hermite_sampling_condition/hermite_condition_number_dft_regular.png")

In [ ]:
# Plot comparison
fig, ax1 = plt.subplots(1, 1, figsize=(10, 6))

# --- Pure time REGULAR: thick translucent underlay + crisp top line ---
ax1.semilogy(
    d_time_dft_finite, kappa_time_dft_finite,
    color='dodgerblue',
    linewidth=8,
    alpha=0.28,
    solid_capstyle='round',
    zorder=1
)

ax1.semilogy(
    d_time_dft_finite, kappa_time_dft_finite,
    color='dodgerblue',
    linestyle='-',
    marker='o',
    markersize=8,
    linewidth=2.6,
    alpha=0.95,
    markerfacecolor='white',
    markeredgecolor='dodgerblue',
    markeredgewidth=1.6,
    zorder=3,
    label='One-sided reconstruction'
)

# --- DFT-stabilized REGULAR: thick translucent underlay + crisp top line ---
ax1.semilogy(
    d_dft_finite, kappa_dft_finite,
    color='orangered',
    linewidth=8,
    alpha=0.28,
    solid_capstyle='round',
    zorder=2
)

ax1.semilogy(
    d_dft_finite, kappa_dft_finite,
    color='orangered',
    linestyle='--',
    marker='s',
    markersize=8,
    linewidth=2.6,
    alpha=0.95,
    markerfacecolor='white',
    markeredgecolor='orangered',
    markeredgewidth=1.6,
    zorder=4,
    label='DFT-induced two-sided reconstruction'
)

ax1.set_xlabel('Dimension $N$', fontsize=23)
ax1.set_ylabel(r'Condition number', fontsize=23)
# ax1.set_title(
#     f'Two-sided reconstruction via applying DFT to half of the samples',
#     fontsize=16
# )
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=18)
ax1.set_xlim([0, max(dimensions) * 1.02])

ax1.tick_params(axis='x', labelsize=16)  # x-axis tick label size
ax1.tick_params(axis='y', labelsize=12)  # y-axis tick label size

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_dft_regular.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# EXPERIMENT 3b: DFT-Stabilizer with RANDOM Sampling
# =============================================================================

results_dft_random = []
output_file_dft_random = Path("results/hermite_sampling_condition/hermite_condition_numbers_dft_random.json")

print(f"EXPERIMENT 3b: DFT-Stabilizer (RANDOM sampling)")
print(f"Computing condition numbers...")
print(f"  Averaging over {NUM_RANDOM_TRIALS} trials (seed={RANDOM_SEED})")
print(f"  TIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}")
print("="*90)

for i, d in enumerate(dimensions):
    k, m = compute_time_freq_split(d, TIME_RATIO)
    
    # Collect condition numbers across all trials
    kappa_time_trials = []
    kappa_dft_trials = []
    
    for trial in range(NUM_RANDOM_TRIALS):
        # Create independent random generators
        trial_offset = trial * 100000
        
        # Pure time gets its own generator
        rng_pure = np.random.default_rng(RANDOM_SEED + d + trial_offset + 30000)
        
        # DFT-stabilized gets its own generator (independent from pure)
        rng_dft = np.random.default_rng(RANDOM_SEED + d + trial_offset + 40000)
        
        # Build pure time sampling matrix with random samples
        A_time_rand = build_pure_time_matrix_random(d, TIME_INTERVAL, rng_pure)
        kappa_time_rand, _, _ = compute_condition_number(A_time_rand, d)
        
        # Build DFT-stabilized matrix with random samples
        A_dft_rand, _, _ = build_dft_stabilized_matrix_random(d, TIME_RATIO, TIME_INTERVAL, rng_dft)
        kappa_dft_rand, _, _ = compute_condition_number(A_dft_rand, d)
        
        # Store trial results
        kappa_time_trials.append(kappa_time_rand if np.isfinite(kappa_time_rand) else np.inf)
        kappa_dft_trials.append(kappa_dft_rand if np.isfinite(kappa_dft_rand) else np.inf)
    
    # Convert to arrays and compute statistics
    kappa_time_trials = np.array(kappa_time_trials)
    kappa_dft_trials = np.array(kappa_dft_trials)
    
    time_finite_trials = kappa_time_trials[np.isfinite(kappa_time_trials)]
    dft_finite_trials = kappa_dft_trials[np.isfinite(kappa_dft_trials)]
    
    # Statistics
    kappa_time_mean = np.mean(time_finite_trials) if len(time_finite_trials) > 0 else np.inf
    kappa_time_std = np.std(time_finite_trials) if len(time_finite_trials) > 1 else 0.0
    kappa_dft_mean = np.mean(dft_finite_trials) if len(dft_finite_trials) > 0 else np.inf
    kappa_dft_std = np.std(dft_finite_trials) if len(dft_finite_trials) > 1 else 0.0
    
    # Store results
    result = {
        "d": d,
        "time_ratio": TIME_RATIO,
        "time_interval": list(TIME_INTERVAL),
        "k": k,
        "m": m,
        "num_trials": NUM_RANDOM_TRIALS,
        "kappa_time_mean": float(kappa_time_mean) if np.isfinite(kappa_time_mean) else "inf",
        "kappa_time_std": float(kappa_time_std) if np.isfinite(kappa_time_std) else "nan",
        "kappa_dft_mean": float(kappa_dft_mean) if np.isfinite(kappa_dft_mean) else "inf",
        "kappa_dft_std": float(kappa_dft_std) if np.isfinite(kappa_dft_std) else "nan",
        "num_finite_time_trials": len(time_finite_trials),
        "num_finite_dft_trials": len(dft_finite_trials),
        "random_seed": RANDOM_SEED
    }
    results_dft_random.append(result)
    
    # Progress update
    if (i + 1) % 20 == 0 or d in [1, 10, 100, 500]:
        kt_str = f"{kappa_time_mean:.4e}" if np.isfinite(kappa_time_mean) else "inf"
        kd_str = f"{kappa_dft_mean:.4e}" if np.isfinite(kappa_dft_mean) else "inf"
        print(f"[{i+1:3d}/{len(dimensions)}] d={d:5d} (k={k}, m={m}): mean kappa_time = {kt_str}, mean kappa_dft = {kd_str}")

# Save results
with open(output_file_dft_random, 'w') as f:
    json.dump(results_dft_random, f, indent=2)

print("\nComputation complete!")
print(f"Results saved to {output_file_dft_random}")

In [ ]:
# Extract random DFT-stabilized data for plotting
d_vals_dft_rand = np.array([r['d'] for r in results_dft_random])
kappa_time_dft_rand_mean = np.array([
    r['kappa_time_mean'] if r['kappa_time_mean'] != 'inf' else np.inf 
    for r in results_dft_random
])
kappa_dft_rand_mean = np.array([
    r['kappa_dft_mean'] if r['kappa_dft_mean'] != 'inf' else np.inf 
    for r in results_dft_random
])
kappa_time_dft_rand_std = np.array([
    r['kappa_time_std'] if r['kappa_time_std'] != 'nan' else 0.0 
    for r in results_dft_random
])
kappa_dft_rand_std = np.array([
    r['kappa_dft_std'] if r['kappa_dft_std'] != 'nan' else 0.0 
    for r in results_dft_random
])

# Filter out infinite values for plotting
time_dft_rand_finite_mask = np.isfinite(kappa_time_dft_rand_mean)
dft_rand_finite_mask = np.isfinite(kappa_dft_rand_mean)

d_time_dft_rand_finite = d_vals_dft_rand[time_dft_rand_finite_mask]
kappa_time_dft_rand_finite = kappa_time_dft_rand_mean[time_dft_rand_finite_mask]
kappa_time_dft_rand_std_finite = kappa_time_dft_rand_std[time_dft_rand_finite_mask]

d_dft_rand_finite = d_vals_dft_rand[dft_rand_finite_mask]
kappa_dft_rand_finite = kappa_dft_rand_mean[dft_rand_finite_mask]
kappa_dft_rand_std_finite = kappa_dft_rand_std[dft_rand_finite_mask]

print(f"DFT-Stabilizer (RANDOM) Results (averaged over {NUM_RANDOM_TRIALS} trials):")
print(f"  Pure time (random): {len(d_time_dft_rand_finite)} finite out of {len(d_vals_dft_rand)}")
print(f"  DFT-stabilized (random): {len(d_dft_rand_finite)} finite out of {len(d_vals_dft_rand)}")

In [ ]:
# Plot DFT-Stabilizer: Regular vs Random
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# -----------------------------------------------------------------------------
# Plot 1 (top-left): Regular DFT-stabilized - Semi-log
# -----------------------------------------------------------------------------
ax1 = axes[0, 0]
ax1.semilogy(d_time_dft_finite, kappa_time_dft_finite, 'b-o', markersize=7, linewidth=1.5,
             alpha=0.8, markeredgecolor='white', markeredgewidth=0.5, zorder=2, 
             label='Pure time REGULAR')
ax1.semilogy(d_dft_finite, kappa_dft_finite, 'g-s', markersize=5, linewidth=1.5,
             alpha=0.9, markeredgecolor='darkgreen', markeredgewidth=0.5, zorder=3,
             label='DFT-stabilized REGULAR')
ax1.set_xlabel('Dimension d', fontsize=11)
ax1.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=11)
ax1.set_title('Regular Sampling: Pure Time vs DFT-Stabilized', fontsize=12, fontweight='bold')
ax1.grid(True, which='both', alpha=0.3)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim([0, max(dimensions) * 1.02])

# -----------------------------------------------------------------------------
# Plot 2 (top-right): Random DFT-stabilized with error bands
# -----------------------------------------------------------------------------
ax2 = axes[0, 1]

# -----------------------------------------------------------------------------
# Plot 2 (top-right): Random DFT-stabilized with error bands
# -----------------------------------------------------------------------------
ax2.semilogy(d_time_dft_rand_finite, kappa_time_dft_rand_finite, 'c-o', markersize=7, linewidth=1.5,
             alpha=0.8, markeredgecolor='white', markeredgewidth=0.5, zorder=2, 
             label=f'Pure time RANDOM (mean, n={NUM_RANDOM_TRIALS})')
if np.any(kappa_time_dft_rand_std_finite > 0):
    ax2.fill_between(d_time_dft_rand_finite, 
                     np.maximum(kappa_time_dft_rand_finite - kappa_time_dft_rand_std_finite, 1e-10),
                     kappa_time_dft_rand_finite + kappa_time_dft_rand_std_finite,
                     alpha=0.2, color='cyan')
ax2.semilogy(d_dft_rand_finite, kappa_dft_rand_finite, 'lime', marker='s', markersize=5, linewidth=1.5,
             alpha=0.9, markeredgecolor='darkgreen', markeredgewidth=0.5, zorder=3,
             label=f'DFT-stabilized RANDOM (mean, n={NUM_RANDOM_TRIALS})')
if np.any(kappa_dft_rand_std_finite > 0):
    ax2.fill_between(d_dft_rand_finite,
                     np.maximum(kappa_dft_rand_finite - kappa_dft_rand_std_finite, 1e-10),
                     kappa_dft_rand_finite + kappa_dft_rand_std_finite,
                     alpha=0.2, color='lime')
ax2.set_xlabel('Dimension d', fontsize=11)
ax2.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=11)
ax2.set_title('Random Sampling: Pure Time vs DFT-Stabilized', fontsize=12, fontweight='bold')
ax2.grid(True, which='both', alpha=0.3)
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xlim([0, max(dimensions) * 1.02])

# -----------------------------------------------------------------------------
# Plot 3 (bottom-left): All 4 curves together - Semi-log
# -----------------------------------------------------------------------------
ax3 = axes[1, 0]
ax3.semilogy(d_time_dft_finite, kappa_time_dft_finite, 'b-o', linewidth=1.5, markersize=6,
             alpha=0.8, markeredgecolor='white', markeredgewidth=0.5, zorder=2, label='Pure time REGULAR')
ax3.semilogy(d_dft_finite, kappa_dft_finite, 'g-s', linewidth=1.5, markersize=5,
             alpha=0.9, markeredgecolor='darkgreen', markeredgewidth=0.5, zorder=3, label='DFT-stab REGULAR')
ax3.semilogy(d_time_dft_rand_finite, kappa_time_dft_rand_finite, 'c--^', linewidth=1.5, markersize=5,
             alpha=0.8, markeredgecolor='white', markeredgewidth=0.5, zorder=4, label='Pure time RANDOM')
ax3.semilogy(d_dft_rand_finite, kappa_dft_rand_finite, 'lime', linestyle='--', marker='D', linewidth=1.5, markersize=4,
             alpha=0.9, markeredgecolor='darkgreen', markeredgewidth=0.5, zorder=5, label='DFT-stab RANDOM')
ax3.set_xlabel('Dimension d', fontsize=11)
ax3.set_ylabel(r'$\kappa_2(A^{(d)})$', fontsize=11)
ax3.set_title('All DFT-Stabilizer Curves (Semi-log)', fontsize=12, fontweight='bold')
ax3.grid(True, which='both', alpha=0.3)
ax3.legend(loc='upper left', fontsize=9)
ax3.set_xlim([0, max(dimensions) * 1.02])

# -----------------------------------------------------------------------------
# Plot 4 (bottom-right): All 4 curves - Log-log
# -----------------------------------------------------------------------------

fig.suptitle(f'EXPERIMENT 3: DFT-Stabilizer\nTIME_RATIO={TIME_RATIO}, TIME_INTERVAL={TIME_INTERVAL}',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('results/hermite_sampling_condition/hermite_condition_number_dft_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved to results/hermite_sampling_condition/hermite_condition_number_dft_comparison.png")

In [ ]:
# =============================================================================
# Summary Statistics for EXPERIMENT 3: DFT-Stabilizer
# =============================================================================

print("="*100)
print("EXPERIMENT 3: DFT-Stabilizer - Summary Statistics")
print("="*100)
print(f"\nConfiguration:")
print(f"  TIME_RATIO = {TIME_RATIO}")
print(f"  TIME_INTERVAL = {TIME_INTERVAL}")
print(f"  NUM_RANDOM_TRIALS = {NUM_RANDOM_TRIALS}")
print(f"  Dimensions tested: {len(dimensions)} (from {min(dimensions)} to {max(dimensions)})")

print(f"\n--- Regular Sampling ---")
print(f"  Pure time: {len(d_time_dft_finite)} finite out of {len(d_vals_dft)}")
if len(kappa_time_dft_finite) > 0:
    print(f"    Min kappa: {np.min(kappa_time_dft_finite):.4e} at d = {d_time_dft_finite[np.argmin(kappa_time_dft_finite)]}")
    print(f"    Max kappa: {np.max(kappa_time_dft_finite):.4e} at d = {d_time_dft_finite[np.argmax(kappa_time_dft_finite)]}")
print(f"  DFT-stabilized: {len(d_dft_finite)} finite out of {len(d_vals_dft)}")
if len(kappa_dft_finite) > 0:
    print(f"    Min kappa: {np.min(kappa_dft_finite):.4e} at d = {d_dft_finite[np.argmin(kappa_dft_finite)]}")
    print(f"    Max kappa: {np.max(kappa_dft_finite):.4e} at d = {d_dft_finite[np.argmax(kappa_dft_finite)]}")

print(f"\n--- Random Sampling (averaged over {NUM_RANDOM_TRIALS} trials) ---")
print(f"  Pure time: {len(d_time_dft_rand_finite)} finite out of {len(d_vals_dft_rand)}")
if len(kappa_time_dft_rand_finite) > 0:
    print(f"    Min mean kappa: {np.min(kappa_time_dft_rand_finite):.4e} at d = {d_time_dft_rand_finite[np.argmin(kappa_time_dft_rand_finite)]}")
    print(f"    Max mean kappa: {np.max(kappa_time_dft_rand_finite):.4e} at d = {d_time_dft_rand_finite[np.argmax(kappa_time_dft_rand_finite)]}")
print(f"  DFT-stabilized: {len(d_dft_rand_finite)} finite out of {len(d_vals_dft_rand)}")
if len(kappa_dft_rand_finite) > 0:
    print(f"    Min mean kappa: {np.min(kappa_dft_rand_finite):.4e} at d = {d_dft_rand_finite[np.argmin(kappa_dft_rand_finite)]}")
    print(f"    Max mean kappa: {np.max(kappa_dft_rand_finite):.4e} at d = {d_dft_rand_finite[np.argmax(kappa_dft_rand_finite)]}")

# Comparison table at key dimensions
print(f"\n--- Comparison at Key Dimensions ---")
print(f"{'d':>5} | {'k':>4} | {'m':>4} | {'Time REG':>12} | {'DFT REG':>12} | {'Time RAND':>12} | {'DFT RAND':>12} | {'Ratio REG':>10} | {'Ratio RAND':>10}")
print("-"*115)

key_dims = [d for d in [1, 2, 5, 10, 20, 50, 100, 200, 500] if d in dimensions]
for kd in key_dims:
    idx_reg = np.where(d_vals_dft == kd)[0]
    idx_rand = np.where(d_vals_dft_rand == kd)[0]
    
    if len(idx_reg) > 0 and len(idx_rand) > 0:
        r_reg = results_dft[idx_reg[0]]
        k, m = r_reg['k'], r_reg['m']
        
        kt_reg = kappa_time_dft_vals[idx_reg[0]]
        kd_reg = kappa_dft_vals[idx_reg[0]]
        kt_rand = kappa_time_dft_rand_mean[idx_rand[0]]
        kd_rand = kappa_dft_rand_mean[idx_rand[0]]
        
        kt_reg_str = f"{kt_reg:.2e}" if np.isfinite(kt_reg) else "inf"
        kd_reg_str = f"{kd_reg:.2e}" if np.isfinite(kd_reg) else "inf"
        kt_rand_str = f"{kt_rand:.2e}" if np.isfinite(kt_rand) else "inf"
        kd_rand_str = f"{kd_rand:.2e}" if np.isfinite(kd_rand) else "inf"
        
        # Compute improvement ratios (time/dft, higher = DFT is better)
        ratio_reg = kt_reg / kd_reg if (np.isfinite(kt_reg) and np.isfinite(kd_reg) and kd_reg > 0) else np.nan
        ratio_rand = kt_rand / kd_rand if (np.isfinite(kt_rand) and np.isfinite(kd_rand) and kd_rand > 0) else np.nan
        
        ratio_reg_str = f"{ratio_reg:.2e}" if np.isfinite(ratio_reg) else "N/A"
        ratio_rand_str = f"{ratio_rand:.2e}" if np.isfinite(ratio_rand) else "N/A"
        
        print(f"{kd:5d} | {k:4d} | {m:4d} | {kt_reg_str:>12} | {kd_reg_str:>12} | {kt_rand_str:>12} | {kd_rand_str:>12} | {ratio_reg_str:>10} | {ratio_rand_str:>10}")

print("\n(Ratio = Pure Time / DFT-stabilized. Ratio > 1 means DFT-stabilized has better condition number.)")